# 07 Epoching

This notebook creates epoched data from cleaned continuous raw derivatives.

Inputs:

- `desc-cleaned_meg.fif`
- project-specific `desc-analysis_events.tsv` if present
- otherwise raw BIDS `events.tsv`
- `config.epochs` for the saved output window
- optional `config.autoreject.tmin/tmax` for a longer autoreject/QC window

Output:

- `epochs/*_desc-cleaned_epo.fif`

Notebook 02 remains optional because this notebook falls back to raw BIDS events when no analysis-event derivative exists.


Autoreject can be fit/applied on a longer window and the cleaned epochs can then be cropped before writing. This is useful for dense note-level paradigms where short saved epochs are desired but artefact detection needs more context.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.epoching import (
    make_epochs_path,
    write_epochs_for_recordings,
)
from meeg_pipeline.workflow import (
    epoching_input_overview_to_dataframe,
    epoching_results_to_dataframe,
    event_coding_preview_to_dataframe,
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Interactive backend for inspection

In [ ]:
%matplotlib qt

mne.viz.set_browser_backend("qt")
mne.set_log_level("WARNING")

print("MNE browser backend:", mne.viz.get_browser_backend())


## Selection

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Single-file/manual inspection cells at the end of notebooks are disabled by default
# so batch runs over many participants do not stop for plots or ad-hoc file views.
RUN_SINGLE_FILE_INSPECTIONS = False
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

Default:

    OVERWRITE_STEPS = []

Existing epoch files are skipped. To recompute epochs:

    OVERWRITE_STEPS = ["epochs"]


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "epochs",
            "overwrite": should_overwrite("epochs", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "epochs",
                OVERWRITE_STEPS,
            ),
        }
    ]
)


## Epoching parameters

In [ ]:
EPOCH_TMIN = config.epochs.tmin
EPOCH_TMAX = config.epochs.tmax
EPOCH_BASELINE = config.epochs.baseline

# Recommended default:
# "trial_type" creates compact MNE event codes from the trial_type column
# and keeps the full event table as metadata.
EVENT_CODE_MODE = "trial_type"

# If true, MNE drops epochs overlapping BAD annotations.
REJECT_BY_ANNOTATION = True

# Optional autoreject.
# This respects configs/local.yaml:
# autoreject:
#   enabled: true
#   use: "Interpolation"
#   tmin: -1.0
#   tmax: 1.0
#   crop_to_epochs: true
if config.autoreject.enabled:
    USE_AUTOREJECT = config.autoreject.use
else:
    USE_AUTOREJECT = None

AUTOREJECT_TMIN = config.autoreject.tmin
AUTOREJECT_TMAX = config.autoreject.tmax
AUTOREJECT_CROP_TO_EPOCHS = config.autoreject.crop_to_epochs
CONSENSUS_PERCS = config.autoreject.consensus_percs
N_INTERPOLATES = config.autoreject.n_interpolates
N_AUTOREJECT_SUBSET = config.autoreject.subset
N_JOBS = config.runtime.n_jobs

pd.DataFrame(
    [
        {
            "saved_tmin": EPOCH_TMIN,
            "saved_tmax": EPOCH_TMAX,
            "baseline": EPOCH_BASELINE,
            "autoreject_enabled": config.autoreject.enabled,
            "autoreject_use": USE_AUTOREJECT,
            "autoreject_tmin": AUTOREJECT_TMIN,
            "autoreject_tmax": AUTOREJECT_TMAX,
            "autoreject_crop_to_epochs": AUTOREJECT_CROP_TO_EPOCHS,
            "event_code_mode": EVENT_CODE_MODE,
            "reject_by_annotation": REJECT_BY_ANNOTATION,
            "consensus_percs": CONSENSUS_PERCS,
            "n_interpolates": N_INTERPOLATES,
            "n_jobs": N_JOBS,
            "autoreject_subset": N_AUTOREJECT_SUBSET,
        }
    ]
)

## Input overview

Event priority:

1. `derivatives/meeg-pipeline/.../events/*_desc-analysis_events.tsv`
2. raw BIDS `*_events.tsv`


In [ ]:
epoching_input_overview_to_dataframe(config, selected_recordings)


## Preview event coding for one recording

In [ ]:
PREVIEW_INDEX = 0

PREVIEW = selected_recordings[PREVIEW_INDEX]

event_coding_preview_to_dataframe(
    config,
    PREVIEW,
    event_code_mode=EVENT_CODE_MODE,
)


## Write epochs

In [ ]:
epochs_policy = existing_output_policy_for_step(
    "epochs",
    OVERWRITE_STEPS,
)

epoch_results = write_epochs_for_recordings(
    config,
    selected_recordings,
    on_existing=epochs_policy,
    tmin=EPOCH_TMIN,
    tmax=EPOCH_TMAX,
    baseline=EPOCH_BASELINE,
    event_code_mode=EVENT_CODE_MODE,
    reject_by_annotation=REJECT_BY_ANNOTATION,
    use_autoreject=USE_AUTOREJECT,
    consensus_percs=CONSENSUS_PERCS,
    n_interpolates=N_INTERPOLATES,
    autoreject_subset=N_AUTOREJECT_SUBSET,
    autoreject_tmin=AUTOREJECT_TMIN,
    autoreject_tmax=AUTOREJECT_TMAX,
    autoreject_crop_to_epochs=AUTOREJECT_CROP_TO_EPOCHS,
    n_jobs=N_JOBS,
)

epoching_results_to_dataframe(selected_recordings, epoch_results)


## Inspect one epochs file

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    INSPECT_SUBJECT = "0001"
    INSPECT_SESSION = None
    INSPECT_TASK = "example"
    INSPECT_RUN = None

    inspect_path = make_epochs_path(
        config,
        subject=INSPECT_SUBJECT,
        session=INSPECT_SESSION,
        task=INSPECT_TASK,
        run=INSPECT_RUN,
    )

    if inspect_path.exists():
        epochs = mne.read_epochs(inspect_path, preload=False, verbose="error")
        inspect_status = pd.DataFrame(
            [
                {
                    "status": "loaded",
                    "n_epochs": len(epochs),
                    "event_id": epochs.event_id,
                    "path": str(inspect_path),
                }
            ]
        )
    else:
        epochs = None
        inspect_status = pd.DataFrame(
            [
                {
                    "status": "missing_input",
                    "n_epochs": 0,
                    "event_id": {},
                    "path": str(inspect_path),
                }
            ]
        )

    inspect_status
else:
    print('Skipped single-file inspection cell 18 in 1B_preprocessing/07_epoching.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Plot inspected epochs

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if "epochs" not in globals() or epochs is None:
        print("No epochs loaded. Run the previous inspection cell first.")
    else:
        epochs.plot(block=True)
else:
    print('Skipped single-file inspection cell 20 in 1B_preprocessing/07_epoching.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Metadata preview

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if "epochs" not in globals() or epochs is None or epochs.metadata is None:
        pd.DataFrame()
    else:
        epochs.metadata.head(30)
else:
    print('Skipped single-file inspection cell 22 in 1B_preprocessing/07_epoching.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')


## Counts by trial type

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_SINGLE_FILE_INSPECTIONS:
    if "epochs" not in globals() or epochs is None or epochs.metadata is None:
        pd.DataFrame()
    else:
        epochs.metadata.groupby("trial_type").size().reset_index(name="n_epochs")
else:
    print('Skipped single-file inspection cell 24 in 1B_preprocessing/07_epoching.ipynb. Set RUN_SINGLE_FILE_INSPECTIONS = True to run it.')
